In [ ]:
"""Day 27 / Session 1 - Lab Assignment 1: Decoding as Distribution Transforms - Temperature Softmax, Entropy & Sampling."""
import numpy as np

logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5])         # raw scores for a 5-token vocab


def stable_softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=float) / temperature  # divide by the temperature first
    z = z - z.max()                                    # then subtract the max so nothing overflows
    e = np.exp(z)
    return e / e.sum()


def entropy(probs):
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]                                       # treat 0 * log(0) as 0
    return float(-np.sum(p * np.log(p)) + 0.0)         # Shannon entropy in nats (+0.0 avoids -0.0)


def greedy_next_token(logits):
    return int(np.argmax(logits))                      # temperature never changes the argmax


def sample_index(probs, u):
    probs = np.asarray(probs, dtype=float)
    cumulative = np.cumsum(probs)
    i = int(np.searchsorted(cumulative, u, side="left"))      # smallest i with cumsum[i] >= u (inverse CDF)
    i = min(i, len(probs) - 1)                                # guard against float round-off at u ~ 1
    while probs[i] == 0 and i < len(probs) - 1:               # a boundary draw never lands on a zero-probability token
        i += 1
    return i


def temperature_entropy(logits, temperatures):
    return np.array([entropy(stable_softmax(logits, T)) for T in temperatures])
